# Lab 08b · Is this image worth sending to a model?

[Open in Colab](https://colab.research.google.com/github/AIGuruOne/OQ-labs/blob/main/notebooks/08b_image_triage.ipynb)

**Day 3 · S16, supplement to labs 08 and 09** · Budget: 15 min · Runs on: Colab or a laptop, **CPU only, no GPU, no model, no API key**

Lab 08 found that tier 3 diagrams — the same drawings as tier 2, photographed from a
printout — scored **0.643** against the vendor model's 1.0. This notebook asks the
question that comes before "which model": *was that image ever readable in the first
place?*

Everything here is measured from the pixels with **numpy and Pillow only**. No model
is called, nothing is downloaded, and it runs in seconds. That is the point: the
cheapest thing in a vision pipeline is the check you do before you spend money.

**A correct result looks like:** a table of five quality measurements over 24 images,
where the measured blur, skew and noise **recover the degradation each image was
built with** (section 5 checks this, and prints PASS or FAIL per measure). Then a
quality gate that rejects the worst images, and the arithmetic for what that gate
saves in model calls.

> All data in this lab is synthetic. No real OQ material anywhere.

## 1. Setup

Nothing to install on Colab: `numpy` and `Pillow` are preinstalled, and both are
already used by labs 08 and 09.

In [ ]:
import io
import math
import sys
from dataclasses import dataclass

import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont

IN_COLAB = "google.colab" in sys.modules
rng = np.random.default_rng(42)          # the whole notebook is deterministic

print(f"runtime  : {'Colab' if IN_COLAB else 'local'} | numpy {np.__version__} | Pillow {Image.__version__}")
print("no model, no network, no GPU")

## 2. The image set, and what was done to each one

The renderer below is the same idea as lab 08's: draw a clean page, then degrade it
in ways a phone photograph degrades a printout — **rotate, blur, add sensor noise,
fade the ink, and save as low-quality JPEG**.

The difference here is that we keep the degradation parameters. They are the ground
truth section 5 checks the measurements against.

In [ ]:
@dataclass(frozen=True)
class Recipe:
    """What we did to an image. Section 5 checks the measurements recover these."""
    item_id: str
    tier: str
    angle: float      # degrees of rotation
    blur: float       # gaussian blur radius, pixels
    noise: float      # gaussian sensor noise, grey levels
    ink: float        # ink strength, 1.0 = full black
    quality: int      # JPEG quality


def draw_page(seed: int) -> Image.Image:
    """A plausible work-order page: a title, a rule, some fields and a small table."""
    im = Image.new("L", (900, 620), 255)
    d = ImageDraw.Draw(im)
    f = ImageFont.load_default()
    r = np.random.default_rng(seed)
    d.text((40, 30), f"WORK ORDER  WO-{118000 + int(r.integers(0, 999)):06d}", fill=0, font=f)
    d.line((40, 52, 860, 52), fill=0, width=2)
    rows = [("Equipment", f"P-{r.integers(1000,1999)}{chr(int(r.integers(65,68)))}"),
            ("Site", str(r.choice(["MRB", "SHZ", "KTF", "TMQ"]))),
            ("Raised", "2026-09-29"), ("Permit", f"PTW-{r.integers(10000,99999)}"),
            ("Discipline", str(r.choice(["Mechanical", "Electrical", "Instrument"])))]
    for i, (k, v) in enumerate(rows):
        y = 80 + i * 34
        d.text((40, y), f"{k}:", fill=0, font=f)
        d.text((220, y), v, fill=0, font=f)
    d.text((40, 280), "READINGS", fill=0, font=f)
    for c, x in enumerate([40, 260, 480, 700]):
        d.line((x, 300, x, 460), fill=0, width=1)
        d.text((x + 8, 306), ["Point", "Before", "After", "Unit"][c], fill=0, font=f)
    for rr in range(4):
        y = 330 + rr * 32
        d.line((40, y, 860, y), fill=0, width=1)
        for c, x in enumerate([40, 260, 480, 700]):
            val = [f"TP-{rr+1}", f"{r.uniform(1,9):.1f}", f"{r.uniform(1,9):.1f}", "mm/s"][c]
            d.text((x + 8, y + 6), val, fill=0, font=f)
    d.line((40, 460, 860, 460), fill=0, width=1)
    d.text((40, 500), "Supervisor sign-off: ______________________", fill=0, font=f)
    return im


def degrade(page: Image.Image, r: Recipe) -> Image.Image:
    """Apply the recipe. Same order as lab 08: rotate, blur, noise, ink, jpeg."""
    im = page.rotate(r.angle, resample=Image.BICUBIC, expand=True, fillcolor=255)
    if r.blur:
        im = im.filter(ImageFilter.GaussianBlur(r.blur))
    arr = np.asarray(im).astype(np.float64)
    arr = 255 - (255 - arr) * r.ink                       # fade the ink
    if r.noise:
        arr = arr + np.random.default_rng(hash(r.item_id) % 2**32).normal(0, r.noise, arr.shape)
    im = Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))
    buf = io.BytesIO()
    im.save(buf, format="JPEG", quality=r.quality)        # lossy, like a phone photo
    return Image.open(io.BytesIO(buf.getvalue())).convert("L")


# Four tiers, six images each. Parameters mirror lab 08's own renderer.
TIERS = {
    "1 scan-clean":  dict(angle=(-0.4, 0.4), blur=0.0, noise=2.0,  ink=1.00, quality=92),
    "2 scan-fair":   dict(angle=(-1.5, 1.5), blur=0.8, noise=9.0,  ink=0.85, quality=45),
    "3 photo-poor":  dict(angle=(-4.0, 4.0), blur=1.4, noise=14.0, ink=0.70, quality=30),
    "4 photo-worst": dict(angle=(-7.0, 7.0), blur=2.2, noise=20.0, ink=0.55, quality=20),
}

RECIPES, IMAGES = [], {}
for t, (tier, spec) in enumerate(TIERS.items()):
    for i in range(6):
        item = f"wo_{t*6 + i + 1:02d}"
        rec = Recipe(item, tier,
                     angle=float(rng.uniform(*spec["angle"])), blur=spec["blur"],
                     noise=spec["noise"], ink=spec["ink"], quality=spec["quality"])
        RECIPES.append(rec)
        IMAGES[item] = degrade(draw_page(seed=1000 + t * 6 + i), rec)

print(f"{len(IMAGES)} images, {len(TIERS)} tiers, {len(IMAGES)//len(TIERS)} each")
print(f"size of the first: {IMAGES['wo_01'].size}")

### Look at them

Two images, best tier and worst. Everything after this is an attempt to say *in
numbers* what you can see here in a second.

In [ ]:
def side_by_side(a: Image.Image, b: Image.Image, scale: float = 0.55) -> Image.Image:
    w = int(max(a.width, b.width) * scale)
    h = int(max(a.height, b.height) * scale)
    out = Image.new("L", (w * 2 + 12, h), 230)
    out.paste(a.resize((w, h)), (0, 0))
    out.paste(b.resize((w, h)), (w + 12, 0))
    return out

print("left: wo_01 (tier 1, clean)      right: wo_24 (tier 4, worst)")
display(side_by_side(IMAGES["wo_01"], IMAGES["wo_24"]))

## 3. Five measurements, from the pixels alone

Each one is a few lines of numpy. None of them needs OpenCV, and none needs a model.

### 3.1 Blur — variance of the Laplacian

The Laplacian is a second-derivative filter: it is large where intensity changes
sharply, near zero on flat areas. A sharp page has strong edges, so the Laplacian
response has **high variance**. Blur the page and the edges smear, the response
flattens, and the variance collapses.

This is the standard blur measure, and it is about four lines.

In [ ]:
def laplacian_variance(im: Image.Image, median: int = 5) -> float:
    """High value = sharp. Collapses towards zero as the image blurs.

    The median filter is not decoration. Sensor grain is high-frequency, and so is a
    sharp edge, so a raw Laplacian cannot tell them apart - a noisy blurred image can
    score HIGHER than a clean blurred one. A median filter removes speckle while
    leaving edges where they are, and the window has to be big enough for the grain:
    the cell below shows 3x3 is NOT enough on tier 4, and 5x5 is.
    """
    if median:
        im = im.filter(ImageFilter.MedianFilter(median))
    a = np.asarray(im, dtype=np.float64)
    # 4-neighbour Laplacian by slicing: centre*4 minus its neighbours
    lap = (4 * a[1:-1, 1:-1] - a[:-2, 1:-1] - a[2:, 1:-1] - a[1:-1, :-2] - a[1:-1, 2:])
    return float(lap.var())


# Mean over each tier, under three amounts of denoising. Sharpness must FALL
# left to right; a column that goes back up is measuring grain, not blur.
print(f"{'tier':16s}{'blur':>7s}{'raw':>10s}{'median 3':>10s}{'median 5':>10s}")
print("-" * 53)
for tier in TIERS:
    items = [r.item_id for r in RECIPES if r.tier == tier]
    blur = next(r.blur for r in RECIPES if r.tier == tier)
    cols = [np.mean([laplacian_variance(IMAGES[i], median=m) for i in items]) for m in (0, 3, 5)]
    print(f"{tier:16s}{blur:6.1f}px" + "".join(f"{c:10.1f}" for c in cols))
print()
print("raw and median-3 both go back UP at tier 4: its grain is being counted as")
print("edges. Median 5 falls all the way down. That is why the default is 5 - chosen")
print("by looking at this table, not by taste.")

### 3.2 Skew — the projection profile

Rotate the page back by a candidate angle, sum each row of ink, and look at how
*spiky* that profile is. When the text lines are horizontal, every line of text
lands in one row: peaks are tall and gaps are empty, so the profile has high
variance. Tilt the page and the ink smears across rows, flattening it.

So: try a range of angles, keep the one with the spikiest profile. That is the skew.

In [ ]:
def estimate_skew(im: Image.Image, limit: float = 9.0, step: float = 0.5) -> float:
    """Degrees the page appears rotated by. Positive = the same sign as Image.rotate."""
    small = im.resize((im.width // 3, im.height // 3))       # 9x less work, same answer
    best_angle, best_score = 0.0, -1.0
    for angle in np.arange(-limit, limit + step, step):
        rotated = small.rotate(-angle, resample=Image.BILINEAR, fillcolor=255)
        ink = 255.0 - np.asarray(rotated, dtype=np.float64)  # ink is bright, paper is 0
        profile = ink.sum(axis=1)                            # one number per row
        score = float(np.diff(profile).var())                # spiky profile = sharp edges
        if score > best_score:
            best_angle, best_score = float(angle), score
    return best_angle


for item in ["wo_01", "wo_08", "wo_15", "wo_24"]:
    r = next(x for x in RECIPES if x.item_id == item)
    print(f"  {item}  rotated by {r.angle:+5.1f} deg  ->  measured {estimate_skew(IMAGES[item]):+5.1f} deg")

### 3.3 Noise — energy that survives a blur

Blur the image slightly and subtract it from itself. Real structure — the strokes of
the text, the table rules — mostly survives a small blur. Sensor noise does not: it
is exactly the high-frequency part that a blur removes.

So the residual is dominated by noise, and a robust spread of that residual
(**median absolute deviation**, not standard deviation, so a few strong edges do not
dominate) estimates the noise level.

In [ ]:
def estimate_noise(im: Image.Image, tile: int = 24) -> float:
    """Sensor noise in grey levels, measured on blank paper only.

    Measuring the whole page does not work: the strokes of the text are a far bigger
    source of variation than the grain, so they drown it. Instead, cut the page into
    tiles, keep the ones with no ink in them - blank margins and gaps between lines -
    and take the median of their standard deviations. Paper should be flat; whatever
    spread is left on it is noise.
    """
    a = np.asarray(im, dtype=np.float64)
    h, w = a.shape
    paper = np.percentile(a, 90)                 # what blank paper reads on this page
    spreads = []
    for y in range(0, h - tile, tile):
        for x in range(0, w - tile, tile):
            patch = a[y:y + tile, x:x + tile]
            if patch.min() > paper - 25:         # no ink in this tile
                spreads.append(patch.std())
    return float(np.median(spreads)) if spreads else 0.0


for item in ["wo_01", "wo_08", "wo_15", "wo_24"]:
    r = next(x for x in RECIPES if x.item_id == item)
    print(f"  {item}  noise applied {r.noise:5.1f}  ->  measured on paper {estimate_noise(IMAGES[item]):5.1f}")

### 3.4 Contrast — how far the ink is from the paper

A faded photocopy has grey text on grey paper. Take the 5th and 95th percentile of
the pixel values: the gap between them is the usable dynamic range. Also count how
much of the image is **clipped** at pure black or pure white, because a photograph
taken under glare loses detail that no model can recover.

In [ ]:
def contrast_stats(im: Image.Image) -> tuple[float, float]:
    """(ink-to-paper separation in grey levels, fraction of pixels clipped).

    The obvious measure - percentile(5) to percentile(95) - is wrong on a document,
    and wrong in a way worth seeing. A page is mostly blank, so BOTH of those
    percentiles land on paper and the "range" comes out near zero however black the
    text is. Ink is a small minority of the pixels, so it has to be measured where it
    lives: the darkest 1%, against paper at the 90th percentile.
    """
    a = np.asarray(im, dtype=np.float64)
    paper = np.percentile(a, 90)
    ink = np.percentile(a, 1)
    clipped = float(((a <= 2) | (a >= 253)).mean())
    return float(paper - ink), clipped


for item in ["wo_01", "wo_08", "wo_15", "wo_24"]:
    r = next(x for x in RECIPES if x.item_id == item)
    rng_, clip = contrast_stats(IMAGES[item])
    naive = float(np.subtract(*np.percentile(np.asarray(IMAGES[item], float), [95, 5])))
    print(f"  {item}  ink {r.ink:.2f}  ->  ink-to-paper {rng_:6.1f}   clipped {clip:5.1%}"
          f"   (the naive 5-95 percentile would say {naive:.1f})")

### 3.5 Effective resolution — is the text big enough to read?

Pixel dimensions lie: a 4000-pixel photograph of a page held at arm's length can have
smaller text than an 800-pixel scan. What matters is the size of the **strokes**.

Estimate it from the ink: the ratio of inked area to the length of its boundary is
roughly the stroke width in pixels. Below about 1.5 px, OCR and vision models start
guessing at characters.

In [ ]:
def stroke_width(im: Image.Image) -> float:
    """Rough mean stroke thickness in pixels. Under ~1.5 px, small text is unreliable."""
    a = np.asarray(im, dtype=np.float64)
    ink = a < (a.mean() - 20)                  # generous threshold: darker than the paper
    if ink.sum() == 0:
        return 0.0
    # boundary pixels: inked, with at least one non-inked 4-neighbour
    p = np.pad(ink, 1, constant_values=False)
    edge = ink & ~(p[:-2, 1:-1] & p[2:, 1:-1] & p[1:-1, :-2] & p[1:-1, 2:])
    return float(2.0 * ink.sum() / max(edge.sum(), 1))


for item in ["wo_01", "wo_24"]:
    print(f"  {item}  stroke width {stroke_width(IMAGES[item]):.2f} px")

## 4. Every image, every measure

One row per image. This is the table a triage step would produce in a real pipeline,
before anything is sent anywhere.

In [ ]:
def measure(item: str) -> dict:
    im = IMAGES[item]
    rng_, clip = contrast_stats(im)
    return {"item": item,
            "sharpness": round(laplacian_variance(im), 1),
            "skew_deg": round(estimate_skew(im), 1),
            "noise": round(estimate_noise(im), 1),
            "range": round(rng_, 1),
            "clipped": round(clip, 4),
            "stroke_px": round(stroke_width(im), 2)}


MEASURED = {r.item_id: measure(r.item_id) for r in RECIPES}

header = f"{'item':7s}{'tier':15s}{'sharp':>9s}{'skew':>7s}{'noise':>7s}{'range':>7s}{'stroke':>8s}"
print(header); print("-" * len(header))
for r in RECIPES:
    m = MEASURED[r.item_id]
    print(f"{r.item_id:7s}{r.tier:15s}{m['sharpness']:9.1f}{m['skew_deg']:7.1f}"
          f"{m['noise']:7.1f}{m['range']:7.1f}{m['stroke_px']:8.2f}")

In [ ]:
# The same table collapsed to one row per tier - the shape is easier to see
print(f"{'tier':15s}{'sharpness':>12s}{'|skew|':>9s}{'noise':>8s}{'range':>8s}{'stroke':>8s}")
print("-" * 60)
TIER_MEANS = {}
for tier in TIERS:
    rows = [MEASURED[r.item_id] for r in RECIPES if r.tier == tier]
    means = dict(sharpness=np.mean([r["sharpness"] for r in rows]),
                 skew=np.mean([abs(r["skew_deg"]) for r in rows]),
                 noise=np.mean([r["noise"] for r in rows]),
                 rng=np.mean([r["range"] for r in rows]),
                 stroke=np.mean([r["stroke_px"] for r in rows]))
    TIER_MEANS[tier] = means
    print(f"{tier:15s}{means['sharpness']:12.1f}{means['skew']:9.2f}{means['noise']:8.1f}"
          f"{means['rng']:8.1f}{means['stroke']:8.2f}")

## 5. Do the measurements recover what we actually did?

This is the part that makes the numbers trustworthy rather than decorative. Each
measure should move monotonically across the tiers, in the direction the degradation
was applied: sharper images have higher Laplacian variance, noisier images measure
noisier, and so on.

If a measure does **not** move monotonically, it is not measuring what its name says,
and it has no business being in a gate.

In [ ]:
def monotonic(values: list[float], direction: str) -> bool:
    diffs = np.diff(values)
    return bool((diffs < 0).all() if direction == "down" else (diffs > 0).all())


tiers = list(TIERS)
CHECKS = {
    "sharpness falls as blur rises": ([TIER_MEANS[t]["sharpness"] for t in tiers], "down"),
    "measured skew rises with rotation": ([TIER_MEANS[t]["skew"] for t in tiers], "up"),
    "measured noise rises with noise": ([TIER_MEANS[t]["noise"] for t in tiers], "up"),
    "ink-to-paper separation falls as ink fades": ([TIER_MEANS[t]["rng"] for t in tiers], "down"),
}

all_ok = True
for label, (values, direction) in CHECKS.items():
    ok = monotonic(values, direction)
    all_ok &= ok
    print(f"  {'PASS' if ok else 'FAIL'}  {label:38s} {[round(v, 1) for v in values]}")

print()
print("every measure recovers the degradation it is named for" if all_ok else
      "at least one measure does not track its degradation - do not gate on it")

In [ ]:
# Per-image, not just per-tier: how close is the measured skew to the applied angle?
errors = [abs(abs(MEASURED[r.item_id]["skew_deg"]) - abs(r.angle)) for r in RECIPES]
print(f"skew: mean absolute error {np.mean(errors):.2f} deg, worst {max(errors):.2f} deg")
print(f"      (the search step is 0.5 deg, so anything under ~0.5 is at the limit of the method)")

## 6. The gate

Now the decision. A gate is a set of thresholds that sorts every image into **send**,
**send with a warning**, or **do not send — re-shoot it**.

The thresholds below are read off the tier table in section 4, not guessed. Change
them and watch the last column move: that is the whole exercise.

In [ ]:
# Every threshold is read off the tier table in section 4. Nothing here is a guess.
GATE = {
    "min_sharpness": 60.0,    # tier 1 ~109, tier 2 ~54: below this, edges are smeared
    "max_skew_deg": 5.0,      # beyond this, table rows stop aligning
    "max_noise": 4.5,         # tier 4 ~4.9: grain starts swallowing thin strokes
    "min_range": 90.0,        # tier 2 ~109, tier 3 ~62: ink approaching paper
    "min_stroke_px": 1.5,     # below this, characters are guesses
}


def verdict(m: dict) -> tuple[str, list[str]]:
    fails = []
    if m["sharpness"] < GATE["min_sharpness"]:   fails.append("blurred")
    if abs(m["skew_deg"]) > GATE["max_skew_deg"]: fails.append("skewed")
    if m["noise"] > GATE["max_noise"]:           fails.append("noisy")
    if m["range"] < GATE["min_range"]:           fails.append("faded")
    if m["stroke_px"] < GATE["min_stroke_px"]:   fails.append("strokes too thin")
    if not fails:
        return "SEND", []
    if len(fails) == 1:
        return "SEND WITH WARNING", fails
    return "RE-SHOOT", fails


print(f"{'item':7s}{'tier':15s}{'verdict':20s}reasons")
print("-" * 74)
VERDICTS = {}
for r in RECIPES:
    v, why = verdict(MEASURED[r.item_id])
    VERDICTS[r.item_id] = v
    print(f"{r.item_id:7s}{r.tier:15s}{v:20s}{', '.join(why)}")

In [ ]:
from collections import Counter
counts = Counter(VERDICTS.values())
print("gate outcome:", dict(counts))
print()
print("Note that min_stroke_px never fires on this set - every image is above 1.5 px.")
print("An inert threshold is not a free one: it has not been calibrated, so it is a")
print("promise you have not tested. Either find data that trips it, or take it out.")
print()
for tier in TIERS:
    c = Counter(VERDICTS[r.item_id] for r in RECIPES if r.tier == tier)
    print(f"  {tier:15s} " + "  ".join(f"{k}={v}" for k, v in sorted(c.items())))

## 7. What the gate is worth

The arithmetic is the argument. A vision model call costs money and latency; a numpy
measurement costs neither. If the gate turns away images that were never going to
produce a usable extraction, it pays for itself immediately — and, more importantly,
it stops a confident wrong answer being written into a system of record.

Change `PER_IMAGE_USD` and `VOLUME` to your own numbers.

In [ ]:
PER_IMAGE_USD = 0.004      # order-of-magnitude for a small vision model on one page
VOLUME = 2000              # images a month
SECONDS_PER_CALL = 2.5

rejected = counts["RE-SHOOT"] / len(RECIPES)
warned = counts["SEND WITH WARNING"] / len(RECIPES)

print(f"images rejected by the gate : {rejected:5.1%}")
print(f"images sent with a warning  : {warned:5.1%}")
print()
print(f"at {VOLUME} images a month and ${PER_IMAGE_USD} a call:")
print(f"  spend without a gate : ${VOLUME * PER_IMAGE_USD:7.2f}")
print(f"  spend with the gate  : ${VOLUME * (1 - rejected) * PER_IMAGE_USD:7.2f}")
print(f"  saved                : ${VOLUME * rejected * PER_IMAGE_USD:7.2f}  "
      f"and {VOLUME * rejected * SECONDS_PER_CALL / 60:.0f} minutes of latency")
print()
print("The money is not the point. The point is that a rejected image comes back as")
print("'photograph this again', which a person can act on - instead of a plausible")
print("extraction with two wrong fields, which nobody notices until it matters.")

## 8. What to take away

**A quality gate is the cheapest stage in a vision pipeline, and the only one that
can say "no".** Every measurement here is a handful of numpy: no model, no GPU, no
network, and it runs in milliseconds per image.

**Measure, do not eyeball.** Section 5 is the part that matters. We checked that each
measure recovers the degradation it is named for, over 24 images, before using any of
them in a gate. A measure that does not track its own degradation has no business
deciding anything.

**Thresholds are a decision, not a constant.** The gate in section 6 rejects images
that lab 08's tier 3 and 4 resemble — and lab 08 measured what that costs: tier 3
scored **0.643** where clean scans scored 1.0. The gate is how you decide whether to
spend a model call on an image like that, or send it back.

**This is what the four fields of a governance pack ask about.** A rejected image is a
wrong answer with a defined path — a person sees it — which is point 3 of the
deployment-ready bar in your capstone brief.

### Where it connects

| | |
|---|---|
| **Lab 08** | scored a vision model on the same four tiers. This notebook explains *why* the lower tiers score worse |
| **Lab 09** | the invented-value problem: a model filling a blank field. A gate is the stage that stops the worst images reaching it |
| **Lab 10** | indexes extractions as text. Anything the gate rejects should never enter that index |
| **Day 5** | the gate, its thresholds and who owns them is a row in the governance pack |

In [ ]:
print("08b IMAGE TRIAGE OK")
print(f"  {len(IMAGES)} images measured on 5 axes, no model calls")
print(f"  ground-truth checks: {'all passed' if all_ok else 'SEE SECTION 5'}")
print(f"  gate: {dict(counts)}")